In [1]:

import pandas as pd
import numpy as np
import joblib

from pathlib import Path
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


In [2]:

BASE_DIR = Path.cwd()

if not (BASE_DIR / "data").exists():
    BASE_DIR = BASE_DIR.parent

DATA_PATH = BASE_DIR / "data" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
MODEL_DIR = BASE_DIR / "models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", BASE_DIR)
print("Dataset path:", DATA_PATH)


Project directory: c:\Users\rupam\OneDrive\Desktop\RetailMind Ai
Dataset path: c:\Users\rupam\OneDrive\Desktop\RetailMind Ai\data\WA_Fn-UseC_-Telco-Customer-Churn.csv


In [3]:

df = pd.read_csv(DATA_PATH)

df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:

print("Shape:", df.shape)

print("\nDataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nTarget Distribution:")
print(df["Churn"].value_counts())


Shape: (7043, 21)

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBill

In [5]:

df.drop(columns=["customerID"], inplace=True)

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

X = df.drop(columns=["Churn"])

y = df["Churn"].map({
    "Yes": 1,
    "No": 0
})

numerical_columns = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical Columns:", numerical_columns)
print("\nCategorical Columns:", categorical_columns)


Numerical Columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical Columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


C:\Users\rupam\AppData\Local\Temp\ipykernel_16588\3733348785.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=["customerID"], inplace=True)
C:\Users\rupam\AppData\Local\Temp\ipykernel_16588\3733348785.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X.select_dtypes(


In [6]:

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_columns),
    ("cat", categorical_pipeline, categorical_columns)
])


In [7]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)


Training data: (5634, 19)
Testing data: (1409, 19)


In [8]:

models = {
    "Decision Tree": DecisionTreeClassifier(
        random_state=42,
        class_weight="balanced"
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ),

    "SVM": SVC(
        kernel="rbf",
        probability=True,
        random_state=42,
        class_weight="balanced"
    )
}


In [9]:

trained_models = {}

for name, classifier in models.items():

    pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", classifier)
    ])

    pipeline.fit(X_train, y_train)

    trained_models[name] = pipeline

    print(name, "trained successfully!")


Decision Tree trained successfully!
Random Forest trained successfully!


c:\Users\rupam\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


SVM trained successfully!


In [10]:

results = []

for name, model in trained_models.items():

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_prob)

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "ROC-AUC": roc_auc
    })

    print(f"\n========== {name} ==========")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")
    print(f"ROC-AUC: {roc_auc:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))



========== Decision Tree ==========
Accuracy: 0.7317
Precision: 0.4946
Recall: 0.4866
F1-score: 0.4906
ROC-AUC: 0.6537

Confusion Matrix:
[[849 186]
 [192 182]]

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.82      0.82      1035
           1       0.49      0.49      0.49       374

    accuracy                           0.73      1409
   macro avg       0.66      0.65      0.65      1409
weighted avg       0.73      0.73      0.73      1409


========== Random Forest ==========
Accuracy: 0.7637
Precision: 0.5485
Recall: 0.6203
F1-score: 0.5822
ROC-AUC: 0.8225

Confusion Matrix:
[[844 191]
 [142 232]]

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.82      0.84      1035
           1       0.55      0.62      0.58       374

    accuracy                           0.76      1409
   macro avg       0.70      0.72      0.71      1409
weighted avg       0.77      

In [11]:

comparison_df = pd.DataFrame(results)

comparison_df = comparison_df.sort_values(
    by=["Recall", "F1-score", "ROC-AUC"],
    ascending=False
).reset_index(drop=True)

comparison_df


,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,SVM,0.743790,0.511504,0.772727,0.615548,0.821065
1,Random Forest,0.763662,0.548463,0.620321,0.582183,0.822457
2,Decision Tree,0.731725,0.494565,0.486631,0.490566,0.653696


In [12]:

best_model_name = comparison_df.iloc[0]["Model"]

best_model = trained_models[best_model_name]

print("Champion Model:", best_model_name)

print("\nChampion Performance:")
print(comparison_df.iloc[0])


Champion Model: SVM

Champion Performance:
Model             SVM
Accuracy      0.74379
Precision    0.511504
Recall       0.772727
F1-score     0.615548
ROC-AUC      0.821065
Name: 0, dtype: object


In [13]:

for name, model in trained_models.items():

    file_name = name.lower().replace(" ", "_") + ".joblib"

    joblib.dump(
        model,
        MODEL_DIR / file_name
    )

joblib.dump(
    best_model,
    MODEL_DIR / "best_churn_model.joblib"
)

comparison_df.to_csv(
    BASE_DIR / "model_comparison.csv",
    index=False
)

print("All models saved successfully!")


All models saved successfully!


In [14]:

customer = {
    "gender": "Male",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure": 12,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "DSL",
    "OnlineSecurity": "Yes",
    "OnlineBackup": "No",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "No",
    "StreamingMovies": "No",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 70.5,
    "TotalCharges": 846.0
}

customer_df = pd.DataFrame([customer])

prediction = best_model.predict(customer_df)[0]

probability = best_model.predict_proba(customer_df)[0][1]

print("Prediction:", "Yes" if prediction == 1 else "No")
print("Churn Probability:", round(probability * 100, 2), "%")


Prediction: Yes
Churn Probability: 37.91 %


In [15]:

readme_path = BASE_DIR / "README.md"

day2_report = f"""

## Day 2 - Model Comparison

| Model | Accuracy | Precision | Recall | F1-score | ROC-AUC |
|---|---:|---:|---:|---:|---:|
"""

for _, row in comparison_df.iterrows():
    day2_report += (
        f"| {row['Model']} "
        f"| {row['Accuracy']:.4f} "
        f"| {row['Precision']:.4f} "
        f"| {row['Recall']:.4f} "
        f"| {row['F1-score']:.4f} "
        f"| {row['ROC-AUC']:.4f} |\n"
    )

day2_report += f"\nChampion Model: {best_model_name}\n"

with open(readme_path, "a", encoding="utf-8") as file:
    file.write(day2_report)

print("README updated successfully!")


README updated successfully!
